# XGBoost Volatility Forecasting Model Development

## Objective
Fine-tune the XGBoost model to minimize RMSE on forward realized volatility forecasts. Development of XGBoost and GARCH are done entirely outside of strategy backtesting period.

## Constraints
- **Training period**: 1993 - 2014
- **Testing period**: 2015 - 2020
- **Strategy Backtesting period**: 2020 - 2025
- **Validation**: Time-series cross-validation (no future data leakage)
- **Target**: Forward 30-day realized volatility

## Outline
1. Data Loading & Preparation
2. Forecast Volatility using XGBoost and GARCH
3. Evaluate model performance against a Naive baseline


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import warnings
from sklearn.metrics import make_scorer
from scipy.stats import spearmanr
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer
from arch import arch_model
from tqdm import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

HORIZON = 30
LOOKBACK = 30

## 1. Data Loading & Preparation


In [2]:
# Load price and VIX data
prices_df = pd.read_parquet('../data/processed/spy_prices.parquet')
prices_df['date'] = pd.to_datetime(prices_df['date'])
prices_df = prices_df.sort_values('date').reset_index(drop=True)

vix_df = pd.read_parquet('../data/processed/vix_data.parquet')
vix_df['date'] = pd.to_datetime(vix_df['date'])
vix_df = vix_df.sort_values('date').reset_index(drop=True)

# Calculate log returns (in percentage terms to match GARCH convention)
prices_df['log_ret'] = np.log(prices_df['close'] / prices_df['close'].shift(1)) * 100

# Initial VIX filter
vix_df = vix_df[vix_df['date'] > pd.Timestamp('1993-01-28')]

# Define cutoff date - NO DATA AFTER THIS FOR TRAINING
TEST_START = pd.Timestamp('2015-01-01')
BACKTEST_START = pd.Timestamp('2020-01-01')

# Split data
no_backtest_df = prices_df[prices_df['date'] < BACKTEST_START].copy()
no_backtest_vix_df = vix_df[vix_df['date'] < BACKTEST_START].copy()

train_df = no_backtest_df[no_backtest_df['date'] < TEST_START].copy()
test_df = no_backtest_df[no_backtest_df['date'] >= TEST_START].copy()
val_df = prices_df[prices_df['date'] >= BACKTEST_START].copy()

vix_train_df = vix_df[vix_df['date'] < TEST_START].copy()
vix_test_df = vix_df[vix_df['date'] >= TEST_START].copy()
vix_val_df = vix_df[vix_df['date'] >= BACKTEST_START].copy()


print(f"Training data: {train_df['date'].min()} to {train_df['date'].max()} ({len(train_df):,} days)")
print(f"Testing data: {test_df['date'].min()} to {test_df['date'].max()} ({len(test_df):,} days)")
print(f"Back-Test data: {val_df['date'].min()} to {val_df['date'].max()} ({len(val_df):,} days)")
print(f"Training data (VIX): {vix_train_df['date'].min()} to {vix_train_df['date'].max()} ({len(vix_train_df):,} days)")
print(f"Testing data (VIX): {vix_test_df['date'].min()} to {vix_test_df['date'].max()} ({len(vix_test_df):,} days)")
print(f"Back-Test data (VIX): {vix_val_df['date'].min()} to {vix_val_df['date'].max()} ({len(vix_val_df):,} days)")

Training data: 1993-01-29 00:00:00 to 2014-12-31 00:00:00 (5,522 days)
Testing data: 2015-01-02 00:00:00 to 2019-12-31 00:00:00 (1,258 days)
Back-Test data: 2020-01-02 00:00:00 to 2025-12-11 00:00:00 (1,495 days)
Training data (VIX): 1993-01-29 00:00:00 to 2014-12-31 00:00:00 (5,522 days)
Testing data (VIX): 2015-01-02 00:00:00 to 2025-12-11 00:00:00 (2,753 days)
Back-Test data (VIX): 2020-01-02 00:00:00 to 2025-12-11 00:00:00 (1,495 days)


## 2. Forecasting Methods (GARCH, XGBoost)


In [3]:
### Define necessary functions

# Loss function weights
under_weight = 1.5
over_weight = 1.0

def create_features(returns: pd.Series, vix: pd.Series, lookback, horizon) -> pd.DataFrame:
    returns = returns.reset_index(drop=True)
    vix = vix.reset_index(drop=True)

    # ===========================================
    # Lagged VIX
    # ===========================================
    yesterday_vix = vix.shift(1)

    # ===========================================
    # REALIZED VOLATILITY FEATURES
    # ===========================================
    rv_1d = (np.abs(returns) * np.sqrt(252)).shift(1)            # Yesterday's |return|
    rv_5d = (returns.rolling(5).std() * np.sqrt(252)).shift(1)   # Past 5 days (t-5 to t-1)
    rv_22d = (returns.rolling(22).std() * np.sqrt(252)).shift(1) # Past 22 days (t-22 to t-1)
    rv_30d = (returns.rolling(30).std() * np.sqrt(252)).shift(1) # Past 30 days (t-30 to t-1)
    rv_60d = (returns.rolling(60).std() * np.sqrt(252)).shift(1) # Past 60 days (t-60 to t-1)
    rv_120d = (returns.rolling(120).std() * np.sqrt(252)).shift(1) # Past 120 days (t-120 to t-1)


    # ===========================================
    # LEVERAGE EFFECT (Black 1976)
    # ===========================================
    ret_1d = returns.shift(1)  # Yesterday's return
    ret_22d = returns.rolling(22).sum().shift(1) # Cumulative return t-22 to t-1
    ret_30d = returns.rolling(30).sum().shift(1) # Cumulative return t-30 to t-1
    ret_60d = returns.rolling(60).sum().shift(1) # Cumulative return t-60 to t-1
    ret_120d = returns.rolling(120).sum().shift(1) # Cumulative return t-120 to t-1
    
    # ===========================================
    # ASYMMETRIC/SIGNED VOLATILITY (Patton & Sheppard 2015)
    # ===========================================
    neg_returns = returns.clip(upper=0)  # Negative returns only
    
    # Realized semivariance: sqrt(sum(r^2)) * sqrt(252) / 100
    rsv_neg_5d = (np.sqrt((neg_returns**2).rolling(5).sum()) * np.sqrt(252)).shift(1)

    # ===========================================
    # BUILD DATAFRAME
    # ===========================================
    df = pd.DataFrame({
        'yesterday_vix': yesterday_vix,
        'rv_1d': rv_1d,
        'rv_5d': rv_5d,
        'rv_22d': rv_22d,       
        'rv_30d': rv_30d,
        'rv_60d': rv_60d,
        'rv_120d': rv_120d,
        'ret_1d': ret_1d,
        'ret_22d': ret_22d,
        'ret_30d': ret_30d,
        'ret_60d': ret_60d,
        'ret_120d': ret_120d,
        'rsv_neg_5d': rsv_neg_5d,
        'horizon': horizon
    })
  
    # Trim to start after lookback (ensures enough history for all features)
    df = df.iloc[lookback:].reset_index(drop=True)
    
    return df

def create_targets(returns: pd.Series, lookback, horizon) -> np.ndarray:
    rolling_vol = returns.rolling(window=horizon).std() * np.sqrt(252)
    targets = rolling_vol.shift(-horizon)
    target_slice = targets.iloc[lookback:-horizon]
    
    return target_slice.values

def create_feature_targets(returns, vix, horizon, lookback):
    # 1. Create Features and Targets for each horizon
    all_X, all_y = [], []
    
    X = create_features(returns, vix, lookback=lookback, horizon=horizon)
    y = create_targets(returns, lookback=lookback, horizon=horizon)
    
    # Align X and y: targets are shorter because they need forward data
    # X has (n - lookback) rows, y has (n - lookback - horizon) rows
    # Trim X from the end to match y
    n_samples = len(y)
    if n_samples > 0:
        X = X.iloc[:n_samples]
        all_X.append(X)
        all_y.append(y)
    
    if len(all_X) == 0:
        raise ValueError("Not enough data to create training samples")
    
    # 2. Combine all horizons
    X_combined = pd.concat(all_X, ignore_index=True)
    y_combined = np.concatenate(all_y)
    
    # 3. Remove rows with NaN values
    mask = ~(X_combined.isna().any(axis=1) | np.isnan(y_combined))
    X_combined = X_combined[mask]
    y_combined = y_combined[mask]

    return X_combined, y_combined

def asymmetric_squared_error(y_true, y_pred):
    """
    Custom XGBoost objective that penalizes underprediction more.
    Underpredicting vol → theo too low → sell signals → dangerous short positions.
    
    Args:
        y_true: Actual forward RV
        y_pred: Model's prediction
        
    Returns:
        gradient, hessian for XGBoost
    """
    residual = y_pred - y_true  # Positive = overprediction, Negative = underprediction
    
    weights = np.where(residual < 0, under_weight, over_weight)
    
    grad = weights * residual
    hess = weights * np.ones_like(residual)
    
    return grad, hess

def asymmetric_rmse(y_true, y_pred):
    residual = y_pred - y_true
    weights = np.where(residual < 0, under_weight, over_weight)  # Penalize underprediction
    
    return -np.sqrt(np.mean(weights * residual**2))  # Negative because sklearn maximizes

def spearman_scorer(y_true, y_pred):
    """Higher is better - captures rank correlation."""
    corr, _ = spearmanr(y_true, y_pred)
    return corr

def fit(returns: pd.Series, vix: pd.Series, min_train_size: int = 30, horizon: int = HORIZON, lookback: int = LOOKBACK) -> tuple:

    if len(returns) < min_train_size:
        raise ValueError(f"Need at least {min_train_size} returns to fit ML model")
    
    # Randomized Search for CV
    model = xgb.XGBRegressor(
        random_state=115,
        objective=asymmetric_squared_error
    )
    param_grid = {
        'n_estimators': [200, 300, 400],
        'max_depth': [3,4,5],
        'learning_rate': [0.005, 0.01, 0.03, 0.05]
    }
    
    # Time-series cross-validation
    tscv = TimeSeriesSplit(n_splits=5)  
    
    regressor = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=500,  # Number of parameter settings to sample
        cv=tscv,
        scoring=make_scorer(spearman_scorer),
        verbose=1,
        random_state=115,
        n_jobs=-1
    )

    # Create features and targets for train data
    X_combined, y_combined = create_feature_targets(returns, vix, lookback=lookback, horizon=horizon)

    # Fit the model
    regressor.fit(X_combined, y_combined)
    
    # Save feature names
    feature_names = X_combined.columns.tolist()
        
    return regressor, feature_names

def evaluate_forecast(y_true, y_pred, name="Model"):
    """Calculate and display forecast metrics."""
    metrics = {
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
        'bias': (y_pred - y_true).mean(),
        'corr': np.corrcoef(y_true, y_pred)[0, 1],
        'spearman': spearmanr(y_true, y_pred)[0],
        'n_samples': len(y_true)
    }
    
    print(f"\n{name}:")
    print(f"  RMSE:        {metrics['rmse']:.6f}")
    print(f"  MAE:         {metrics['mae']:.6f}")
    print(f"  Bias:        {metrics['bias']:+.6f}")
    print(f"  Correlation: {metrics['corr']:.4f}")
    print(f"  Spearman:    {metrics['spearman']:.4f}")
    print(f"  Samples:     {metrics['n_samples']:,}")
    
    return metrics



In [4]:
# ===========================================
# GARCH ROLLING FORECAST ON TEST SET (2015-2020)
# ===========================================

# Combine train and test for rolling window
test_returns = test_df['log_ret'].dropna()
test_dates = test_df[test_df['log_ret'].notna()]['date']

# Parameters
HORIZON = 30
REFIT_FREQ = 1  # Refit GARCH every day

# Storage for GARCH predictions
garch_forecasts_test = []
garch_dates_test = []
garch_actuals_test = create_targets(test_returns, lookback=LOOKBACK, horizon=HORIZON)

# GARCH model state
garch_result = None
omega, alpha, beta = None, None, None
current_var = None

for i in tqdm(range(LOOKBACK, len(test_returns) - HORIZON), desc="GARCH Rolling"):
    current_date = test_dates.iloc[i]
    
    # Refit GARCH periodically or on first iteration
    if garch_result is None or (i - LOOKBACK) % REFIT_FREQ == 0:
        try:
            # Fit GARCH on all data up to current point
            returns_to_fit = test_returns.iloc[:i]
            garch_model = arch_model(returns_to_fit, vol='GARCH', p=1, q=1, dist='t')
            garch_result = garch_model.fit(disp='off', show_warning=False)
            
            # Extract parameters
            omega = garch_result.params['omega']
            alpha = garch_result.params['alpha[1]']
            beta = garch_result.params['beta[1]']
            current_var = garch_result.conditional_volatility.iloc[-1] ** 2
        except Exception as e:
            continue
    else:
        # One-step variance update using yesterday's return
        prev_return = test_returns.iloc[i-1]
        current_var = omega + alpha * (prev_return ** 2) + beta * current_var
    
    # Multi-step forecast: E[sigma^2_{t+h}] using GARCH recursion
    persistence = alpha + beta
    if persistence >= 0.9999:
        # Near unit root - use current variance
        avg_var_forecast = current_var
    else:
        # Mean-reverting GARCH: forecast converges to unconditional variance
        long_run_var = omega / (1 - persistence)
        # Average variance over next H days (integrated forecast)
        avg_var_forecast = long_run_var + (1 - persistence**HORIZON) / (HORIZON * (1 - persistence)) * (current_var - long_run_var)
    
    # Convert to annualized vol (returns are in %, so sqrt(var*252) gives annualized %)
    garch_vol_forecast = np.sqrt(avg_var_forecast * 252)
    
    # Calculate actual forward RV: std of next HORIZON returns, annualized
    forward_returns = test_returns.iloc[i:i+HORIZON]
    
    garch_forecasts_test.append(garch_vol_forecast)
    garch_dates_test.append(current_date)

print(f"\nGARCH: Generated {len(garch_forecasts_test)} forecasts")
print(f"Date range: {garch_dates_test[0]} to {garch_dates_test[-1]}")

GARCH Rolling: 100%|██████████| 1198/1198 [00:31<00:00, 38.47it/s]


GARCH: Generated 1198 forecasts
Date range: 2015-02-17 00:00:00 to 2019-11-15 00:00:00


In [5]:
# ===========================================
# Fit and forcast using XGBoost model
# ===========================================
print("Fitting XGBoost model...")
returns = train_df['log_ret'].dropna()
vix_train = vix_train_df['vix_close'].dropna()
vix_test = vix_test_df['vix_close'].dropna()

# Fit with default model
fitted_model, feature_cols = fit(returns, vix_train)

print(f"\nModel fitted successfully!")
print(f"Feature columns: {feature_cols}")
print(f"Best Parameters found: {fitted_model.best_params_}")

# Store best parameters
BEST_PARAMS = fitted_model.best_params_.copy()
BEST_PARAMS['objective'] = asymmetric_squared_error
BEST_PARAMS['random_state'] = 115

# Create train and test features
X_train, y_train = create_feature_targets(train_df['log_ret'].dropna(), vix_train, HORIZON, LOOKBACK)
X_test, y_test = create_feature_targets(test_df['log_ret'].dropna(), vix_test, HORIZON, LOOKBACK)

# Train best model on full training data
best_model = xgb.XGBRegressor(**BEST_PARAMS)
best_model.fit(X_train, y_train)

# Run best model
test_preds = best_model.predict(X_test)

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nSorted Features:")
display(importance)

Fitting XGBoost model...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Model fitted successfully!
Feature columns: ['yesterday_vix', 'rv_1d', 'rv_5d', 'rv_22d', 'rv_30d', 'rv_60d', 'rv_120d', 'ret_1d', 'ret_22d', 'ret_30d', 'ret_60d', 'ret_120d', 'rsv_neg_5d', 'horizon']
Best Parameters found: {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.01}

Sorted Features:


,feature,importance
4,rv_30d,0.312636
10,ret_60d,0.148374
5,rv_60d,0.116191
12,rsv_neg_5d,0.101168
0,yesterday_vix,0.098902
6,rv_120d,0.049183
3,rv_22d,0.046286
11,ret_120d,0.036037
2,rv_5d,0.030499
9,ret_30d,0.029936


## 3. GARCH vs XGBoost vs Naive Comparison

Compare performance of all three forecasting methods on the same test period **(2015-2020)**.

### Methodology
- **Naive**: Trailing 30-day RV (assumes volatility persists unchanged)
- **GARCH**: Rolling refit every day, forecast 30 days ahead using mean-reversion
- **XGBoost**: Single model trained on training data, forecast on test set
- **Target**: Forward 30-day realized volatility


In [6]:
# Get the mask for non-NaN rows
mask = ~(X_test.isna().any(axis=1) | np.isnan(y_test))

# Dates start at LOOKBACK, go for n_samples, then filter by mask
test_dates_all = test_df[test_df['log_ret'].notna()]['date'].reset_index(drop=True)
xgb_dates = test_dates_all.iloc[LOOKBACK:LOOKBACK + len(y_test)][mask.values].reset_index(drop=True)

# Naive forecast: trailing 30-day RV (rv_30d feature)
naive_preds = X_test['rv_30d'].values

# Build XGBoost dataframe (include naive forecast)
xgb_df = pd.DataFrame({
    'date': pd.to_datetime(xgb_dates),
    'xgb_pred': test_preds,
    'naive_pred': naive_preds,
    'actual': y_test
})

# Build GARCH dataframe
garch_df = pd.DataFrame({
    'date': pd.to_datetime(garch_dates_test),
    'garch_pred': garch_forecasts_test,
    'garch_actual': garch_actuals_test
})

# Combine into one dataframe
comparison_df = pd.merge(xgb_df, garch_df, on='date', how='inner', suffixes=('_xgb', '_garch'))

# Ensure dates are aligned through actuals check
assert comparison_df['actual'].all() == comparison_df['garch_actual'].all() 

# Use one actuals column
comparison_df['actual_rv'] = comparison_df['actual']
comparison_df = comparison_df.drop(columns=['actual', 'garch_actual'])

print(f"\nComparison dataset: {len(comparison_df)} matched observations")
print(f"Date range: {comparison_df['date'].min()} to {comparison_df['date'].max()}")
comparison_df.sample(5)



Comparison dataset: 1108 matched observations
Date range: 2015-02-17 00:00:00 to 2019-07-11 00:00:00


,date,xgb_pred,naive_pred,garch_pred,actual_rv
274,2016-03-18,13.338076,16.652320,14.790812,5.087272
497,2017-02-06,10.443008,7.206900,12.014253,7.305893
1039,2019-04-03,12.792825,14.205520,15.340966,16.631870
885,2018-08-21,29.567066,26.991765,12.091166,15.966124
821,2018-05-21,10.410044,6.725165,12.328058,21.158687


In [7]:
# ===========================================
# Evaluate model results
# ===========================================

def calculate_all_metrics(y_true, y_pred, name):
    """Calculate comprehensive forecast metrics."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    bias = (y_pred - y_true).mean()
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    spearman_corr, _ = spearmanr(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    # Under/over prediction rates
    under_pred_rate = (y_pred < y_true).mean() * 100
    over_pred_rate = (y_pred > y_true).mean() * 100
    
    return {
        'Model': name,
        'RMSE': rmse,
        'MAE': mae,
        'Bias': bias,
        'Correlation': corr,
        'Spearman': spearman_corr,
        'MAPE (%)': mape,
        'Under-Pred (%)': under_pred_rate,
        'Over-Pred (%)': over_pred_rate
    }

# Calculate metrics for all three models
naive_metrics = calculate_all_metrics(
    comparison_df['actual_rv'].values, 
    comparison_df['naive_pred'].values, 
    'Naive (30d RV)'
)
garch_metrics = calculate_all_metrics(
    comparison_df['actual_rv'].values, 
    comparison_df['garch_pred'].values, 
    'GARCH(1,1)'
)
xgb_metrics = calculate_all_metrics(
    comparison_df['actual_rv'].values, 
    comparison_df['xgb_pred'].values, 
    'XGBoost'
)

# Create comparison table
metrics_df = pd.DataFrame([naive_metrics, garch_metrics, xgb_metrics]).set_index('Model')

print("="*70)
print("MODEL COMPARISON: Naive vs GARCH(1,1) vs XGBoost")
print(f"Test Period: {comparison_df['date'].min().strftime('%Y-%m-%d')} to {comparison_df['date'].max().strftime('%Y-%m-%d')}")
print(f"Horizon: {HORIZON} days | Observations: {len(comparison_df):,}")
print("="*70)

display(metrics_df.round(4).T)

# Calculate improvements over naive baseline
print("\n" + "="*70)
print("IMPROVEMENT OVER NAIVE BASELINE")
print("="*70)
for model_name, model_metrics in [('GARCH(1,1)', garch_metrics), ('XGBoost', xgb_metrics)]:
    rmse_improv = (naive_metrics['RMSE'] - model_metrics['RMSE']) / naive_metrics['RMSE'] * 100
    mae_improv = (naive_metrics['MAE'] - model_metrics['MAE']) / naive_metrics['MAE'] * 100
    corr_gain = model_metrics['Correlation'] - naive_metrics['Correlation']
    spearman_gain = model_metrics['Spearman'] - naive_metrics['Spearman']
    
    print(f"\n{model_name}:")
    print(f"  RMSE Reduction:      {rmse_improv:+.2f}%")
    print(f"  MAE Reduction:       {mae_improv:+.2f}%")
    print(f"  Correlation Gain:    {corr_gain:+.4f}")
    print(f"  Spearman Gain:       {spearman_gain:+.4f}")

# XGBoost vs GARCH comparison
print("\n" + "="*70)
print("XGBoost vs GARCH(1,1)")
print("="*70)
rmse_improv = (garch_metrics['RMSE'] - xgb_metrics['RMSE']) / garch_metrics['RMSE'] * 100
mae_improv = (garch_metrics['MAE'] - xgb_metrics['MAE']) / garch_metrics['MAE'] * 100
print(f"  RMSE Reduction:      {rmse_improv:+.2f}%")
print(f"  MAE Reduction:       {mae_improv:+.2f}%")
print(f"  Correlation Gain:    {xgb_metrics['Correlation'] - garch_metrics['Correlation']:+.4f}")
print(f"  Spearman Gain:       {xgb_metrics['Spearman'] - garch_metrics['Spearman']:+.4f}")

MODEL COMPARISON: Naive vs GARCH(1,1) vs XGBoost
Test Period: 2015-02-17 to 2019-07-11
Horizon: 30 days | Observations: 1,108


Model,Naive (30d RV),"GARCH(1,1)",XGBoost
RMSE,6.8651,8.3102,6.5480
MAE,5.0366,6.7889,5.3677
Bias,0.1435,1.8272,1.7989
Correlation,0.3527,-0.1439,0.3311
Spearman,0.4425,-0.1651,0.4099
MAPE (%),41.9809,68.0088,50.5429
Under-Pred (%),45.2166,34.2058,29.0614
Over-Pred (%),54.7834,65.7942,70.9386



IMPROVEMENT OVER NAIVE BASELINE

GARCH(1,1):
  RMSE Reduction:      -21.05%
  MAE Reduction:       -34.79%
  Correlation Gain:    -0.4966
  Spearman Gain:       -0.6075

XGBoost:
  RMSE Reduction:      +4.62%
  MAE Reduction:       -6.57%
  Correlation Gain:    -0.0216
  Spearman Gain:       -0.0325

XGBoost vs GARCH(1,1)
  RMSE Reduction:      +21.21%
  MAE Reduction:       +20.93%
  Correlation Gain:    +0.4750
  Spearman Gain:       +0.5750


In [8]:
# ===========================================
# VISUAL COMPARISON: Naive vs GARCH vs XGBoost
# ===========================================

# Calculate errors for histograms
comparison_df['naive_errors'] = comparison_df['naive_pred'] - comparison_df['actual_rv']
comparison_df['garch_errors'] = comparison_df['garch_pred'] - comparison_df['actual_rv']
comparison_df['xgb_errors'] = comparison_df['xgb_pred'] - comparison_df['actual_rv']

# Create subplots: 2 rows, 3 cols
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        f"Naive: Corr={naive_metrics['Correlation']:.3f}, RMSE={naive_metrics['RMSE']:.2f}",
        f"GARCH: Corr={garch_metrics['Correlation']:.3f}, RMSE={garch_metrics['RMSE']:.2f}",
        f"XGBoost: Corr={xgb_metrics['Correlation']:.3f}, RMSE={xgb_metrics['RMSE']:.2f}",
        "Time Series (click legend to toggle models)",
        "Error Distributions",
        "RMSE Comparison"
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.06
)

# Perfect line data
min_rv, max_rv = comparison_df['actual_rv'].min(), comparison_df['actual_rv'].max()

# Row 1: Scatter plots
# Naive scatter
fig.add_trace(go.Scatter(x=comparison_df['actual_rv'], y=comparison_df['naive_pred'],
    mode='markers', marker=dict(size=5, opacity=0.4, color='#1f77b4'),
    name='Naive', showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=[min_rv, max_rv], y=[min_rv, max_rv],
    mode='lines', line=dict(dash='dash', color='black', width=2),
    name='Perfect', showlegend=False), row=1, col=1)

# GARCH scatter
fig.add_trace(go.Scatter(x=comparison_df['actual_rv'], y=comparison_df['garch_pred'],
    mode='markers', marker=dict(size=5, opacity=0.4, color='#ff7f0e'),
    name='GARCH', showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=[min_rv, max_rv], y=[min_rv, max_rv],
    mode='lines', line=dict(dash='dash', color='black', width=2),
    showlegend=False), row=1, col=2)

# XGBoost scatter
fig.add_trace(go.Scatter(x=comparison_df['actual_rv'], y=comparison_df['xgb_pred'],
    mode='markers', marker=dict(size=5, opacity=0.4, color='#2ca02c'),
    name='XGBoost', showlegend=False), row=1, col=3)
fig.add_trace(go.Scatter(x=[min_rv, max_rv], y=[min_rv, max_rv],
    mode='lines', line=dict(dash='dash', color='black', width=2),
    showlegend=False), row=1, col=3)

# Row 2, Col 1: Time series (interactive legend - click to toggle!)
fig.add_trace(go.Scatter(x=comparison_df['date'], y=comparison_df['actual_rv'],
    mode='lines', line=dict(color='black', width=2),
    name='Actual RV'), row=2, col=1)
fig.add_trace(go.Scatter(x=comparison_df['date'], y=comparison_df['naive_pred'],
    mode='lines', line=dict(color='#1f77b4', width=1.5),
    name='Naive'), row=2, col=1)
fig.add_trace(go.Scatter(x=comparison_df['date'], y=comparison_df['garch_pred'],
    mode='lines', line=dict(color='#ff7f0e', width=1.5),
    name='GARCH'), row=2, col=1)
fig.add_trace(go.Scatter(x=comparison_df['date'], y=comparison_df['xgb_pred'],
    mode='lines', line=dict(color='#2ca02c', width=1.5),
    name='XGBoost'), row=2, col=1)

# Row 2, Col 2: Error histograms
fig.add_trace(go.Histogram(x=comparison_df['naive_errors'], name='Naive Err',
    marker_color='#1f77b4', opacity=0.5, nbinsx=40, showlegend=False), row=2, col=2)
fig.add_trace(go.Histogram(x=comparison_df['garch_errors'], name='GARCH Err',
    marker_color='#ff7f0e', opacity=0.5, nbinsx=40, showlegend=False), row=2, col=2)
fig.add_trace(go.Histogram(x=comparison_df['xgb_errors'], name='XGB Err',
    marker_color='#2ca02c', opacity=0.5, nbinsx=40, showlegend=False), row=2, col=2)
fig.add_vline(x=0, line_dash="dash", line_color="red", line_width=2, row=2, col=2)

# Row 2, Col 3: RMSE bar chart
models = ['Naive', 'GARCH', 'XGBoost']
rmses = [naive_metrics['RMSE'], garch_metrics['RMSE'], xgb_metrics['RMSE']]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
fig.add_trace(go.Bar(x=models, y=rmses, marker_color=colors,
    text=[f'{r:.2f}' for r in rmses], textposition='outside',
    showlegend=False), row=2, col=3)

# Update axes labels
fig.update_xaxes(title_text="Actual RV (%)", row=1, col=1)
fig.update_xaxes(title_text="Actual RV (%)", row=1, col=2)
fig.update_xaxes(title_text="Actual RV (%)", row=1, col=3)
fig.update_yaxes(title_text="Predicted RV (%)", row=1, col=1)
fig.update_yaxes(title_text="Predicted RV (%)", row=1, col=2)
fig.update_yaxes(title_text="Predicted RV (%)", row=1, col=3)
fig.update_xaxes(title_text="Date", row=2, col=1)
fig.update_yaxes(title_text="Volatility (%)", row=2, col=1)
fig.update_xaxes(title_text="Forecast Error", row=2, col=2)
fig.update_yaxes(title_text="Count", row=2, col=2)
fig.update_yaxes(title_text="RMSE", row=2, col=3)

# Layout
fig.update_layout(
    height=700, width=1400,
    title_text="Model Comparison: Naive vs GARCH vs XGBoost",
    barmode='overlay',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)

fig.show()


In [9]:
# Define volatility regimes based on actual realized vol
comparison_df['vol_regime'] = pd.cut(
    comparison_df['actual_rv'] / 100,  # Convert to decimal for regime cuts
    bins=[0, 0.12, 0.25, 1.0],
    labels=['Low (<12%)', 'Normal (12-25%)', 'Stress (>25%)']
)

# Calculate errors (naive errors already computed in previous cell)
comparison_df['naive_error'] = comparison_df['naive_pred'] - comparison_df['actual_rv']
comparison_df['garch_error'] = comparison_df['garch_pred'] - comparison_df['actual_rv']
comparison_df['xgb_error'] = comparison_df['xgb_pred'] - comparison_df['actual_rv']
comparison_df['naive_abs_error'] = comparison_df['naive_error'].abs()
comparison_df['garch_abs_error'] = comparison_df['garch_error'].abs()
comparison_df['xgb_abs_error'] = comparison_df['xgb_error'].abs()

# Metrics by regime
regime_metrics = []
for regime in comparison_df['vol_regime'].unique():
    if pd.isna(regime):
        continue
    subset = comparison_df[comparison_df['vol_regime'] == regime]
    
    regime_metrics.append({
        'Regime': regime,
        'N': len(subset),
        'Naive RMSE': np.sqrt((subset['naive_error']**2).mean()),
        'GARCH RMSE': np.sqrt((subset['garch_error']**2).mean()),
        'XGB RMSE': np.sqrt((subset['xgb_error']**2).mean()),
        'Naive Bias': subset['naive_error'].mean(),
        'GARCH Bias': subset['garch_error'].mean(),
        'XGB Bias': subset['xgb_error'].mean()
    })

regime_df = pd.DataFrame(regime_metrics)

print("="*70)
print("PERFORMANCE BY VOLATILITY REGIME")
print("="*70)
display(regime_df.round(2))

# Pairwise win rates
# XGBoost vs Naive
xgb_beats_naive = (comparison_df['xgb_abs_error'] < comparison_df['naive_abs_error']).sum()
# XGBoost vs GARCH
xgb_beats_garch = (comparison_df['xgb_abs_error'] < comparison_df['garch_abs_error']).sum()
# GARCH vs Naive
garch_beats_naive = (comparison_df['garch_abs_error'] < comparison_df['naive_abs_error']).sum()

n = len(comparison_df)
print(f"\nPairwise Win Rates (by observation):")
print(f"  XGBoost beats Naive:  {xgb_beats_naive}/{n} ({xgb_beats_naive/n*100:.1f}%)")
print(f"  XGBoost beats GARCH:  {xgb_beats_garch}/{n} ({xgb_beats_garch/n*100:.1f}%)")
print(f"  GARCH beats Naive:    {garch_beats_naive}/{n} ({garch_beats_naive/n*100:.1f}%)")


PERFORMANCE BY VOLATILITY REGIME


,Regime,N,Naive RMSE,GARCH RMSE,XGB RMSE,Naive Bias,GARCH Bias,XGB Bias
0,Normal (12-25%),411,8.30,7.72,6.89,-2.68,-3.44,-1.37
1,Low (<12%),648,5.22,7.99,5.57,2.72,6.42,4.77
2,Stress (>25%),49,11.31,14.79,12.70,-10.20,-14.71,-10.93



Pairwise Win Rates (by observation):
  XGBoost beats Naive:  467/1108 (42.1%)
  XGBoost beats GARCH:  701/1108 (63.3%)
  GARCH beats Naive:    386/1108 (34.8%)


In [10]:
# ===========================================
# STATISTICAL TESTS: Diebold-Mariano for all pairs
# ===========================================

def qlike(realized_vol, pred):
    """QLIKE loss function for volatility forecasts."""
    realized_vol = np.asarray(realized_vol)
    pred = np.asarray(pred)
    
    # Full formula: sigma^2/h - log(sigma^2/h) - 1
    return realized_vol / pred - np.log(realized_vol / pred) - 1

def diebold_mariano_test(realized_vol, pred1, pred2, h=30):
    """
    Diebold-Mariano test for equal predictive accuracy.
    
    H0: No difference in predictive accuracy
    H1: pred1 has different accuracy than pred2
    
    Args:
        realized_vol: Actual values
        pred1: Predictions from model 1
        pred2: Predictions from model 2
        h: forecast horizon (for Newey-West bandwidth)
    
    Returns:
        dm_stat: DM test statistic (negative = pred1 is better)
        p_value: Two-sided p-value
    """
    # Use Q-likelihood loss for volatility forecasts
    d = qlike(realized_vol, pred1) - qlike(realized_vol, pred2)
    
    T = len(d)
    d_bar = np.mean(d)
    
    # Newey-West HAC variance estimate (bandwidth L = h - 1)
    d_centered = d - d_bar
    gamma_0 = np.mean(d_centered**2)
    
    # γ_k for k = 1, ..., h-1 with Bartlett weights
    gamma_sum = 0.0
    for k in range(1, h):
        gamma_k = np.mean(d_centered[k:] * d_centered[:-k])
        w_k = 1 - k / h  # Bartlett kernel
        gamma_sum += 2 * w_k * gamma_k
    
    lrv = gamma_0 + gamma_sum  # long-run variance estimate
    var_d_bar = lrv / T        # variance of d̄
    
    # DM statistic
    dm_stat = d_bar / np.sqrt(var_d_bar)
    
    # p-value (asymptotic normal)
    p_value = 2 * (1 - stats.norm.cdf(np.abs(dm_stat)))
    
    return dm_stat, p_value

# Calculate QLIKE for all models
T = len(comparison_df)
naive_qlike = qlike(comparison_df['actual_rv'], comparison_df['naive_pred'])
garch_qlike = qlike(comparison_df['actual_rv'], comparison_df['garch_pred'])
xgb_qlike = qlike(comparison_df['actual_rv'], comparison_df['xgb_pred'])

print("="*70)
print("QLIKE LOSS COMPARISON (lower is better)")
print("="*70)
print(f"Sample size: {T}")
print(f"  Naive (30d RV):  {naive_qlike.mean():.6f}")
print(f"  GARCH(1,1):      {garch_qlike.mean():.6f}")
print(f"  XGBoost:         {xgb_qlike.mean():.6f}")

# Run DM tests for all pairs
print("\n" + "="*70)
print("DIEBOLD-MARIANO TESTS (QLIKE loss)")
print("H0: No difference in predictive accuracy")
print("Negative DM stat = first model is better")
print("="*70)

# XGBoost vs Naive
dm_xgb_naive, p_xgb_naive = diebold_mariano_test(
    comparison_df['actual_rv'], comparison_df['xgb_pred'], comparison_df['naive_pred']
)
print(f"\nXGBoost vs Naive:")
print(f"  DM statistic: {dm_xgb_naive:.4f}")
print(f"  p-value:      {p_xgb_naive:.4f}")
print(f"  Result:       {'XGBoost significantly better' if p_xgb_naive < 0.05 and dm_xgb_naive < 0 else 'Naive significantly better' if p_xgb_naive < 0.05 and dm_xgb_naive > 0 else 'No significant difference'}")

# GARCH vs Naive
dm_garch_naive, p_garch_naive = diebold_mariano_test(
    comparison_df['actual_rv'], comparison_df['garch_pred'], comparison_df['naive_pred']
)
print(f"\nGARCH vs Naive:")
print(f"  DM statistic: {dm_garch_naive:.4f}")
print(f"  p-value:      {p_garch_naive:.4f}")
print(f"  Result:       {'GARCH significantly better' if p_garch_naive < 0.05 and dm_garch_naive < 0 else 'Naive significantly better' if p_garch_naive < 0.05 and dm_garch_naive > 0 else 'No significant difference'}")

# XGBoost vs GARCH
dm_xgb_garch, p_xgb_garch = diebold_mariano_test(
    comparison_df['actual_rv'], comparison_df['xgb_pred'], comparison_df['garch_pred']
)
print(f"\nXGBoost vs GARCH:")
print(f"  DM statistic: {dm_xgb_garch:.4f}")
print(f"  p-value:      {p_xgb_garch:.4f}")
print(f"  Result:       {'XGBoost significantly better' if p_xgb_garch < 0.05 and dm_xgb_garch < 0 else 'GARCH significantly better' if p_xgb_garch < 0.05 and dm_xgb_garch > 0 else 'No significant difference'}")

QLIKE LOSS COMPARISON (lower is better)
Sample size: 1108
  Naive (30d RV):  0.146364
  GARCH(1,1):      0.169654
  XGBoost:         0.115594

DIEBOLD-MARIANO TESTS (QLIKE loss)
H0: No difference in predictive accuracy
Negative DM stat = first model is better

XGBoost vs Naive:
  DM statistic: -1.1336
  p-value:      0.2570
  Result:       No significant difference

GARCH vs Naive:
  DM statistic: 0.6162
  p-value:      0.5378
  Result:       No significant difference

XGBoost vs GARCH:
  DM statistic: -2.8518
  p-value:      0.0043
  Result:       XGBoost significantly better
